# Heady™ Vector Embeddings & Memory
## Python SDK — Interactive Notebook

© 2024-2026 HeadySystems Inc. All Rights Reserved.

This notebook demonstrates:
- **384-dimensional vector space operations** (dot product, cosine similarity, PCA)
- **RAM-first vector memory** with namespace isolation
- **Semantic search** across stored embeddings
- **Drift detection** for monitoring embedding quality
- **Persistence** via JSON-lines format
- **PCA dimensionality reduction** via power iteration

---

## Setup

In [ ]:
!pip install numpy matplotlib plotly sentence-transformers -q

import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../python')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.vector_space_ops import (
    EMBEDDING_DIM, dot_product, magnitude, normalize,
    cosine_similarity, euclidean_distance, add, subtract,
    scale, centroid, lerp, random_vector, pca,
)
from core.vector_memory import VectorMemory, DRIFT_THRESHOLD

rng = np.random.default_rng(42)
print(f'Heady™ Vector SDK loaded — EMBEDDING_DIM={EMBEDDING_DIM}')

---
## 1. Vector Space Primitives

Core operations on 384-dimensional vectors — the building blocks of Heady's brain.

In [ ]:
# Generate random unit vectors in 384D
v1 = random_vector(rng=rng)
v2 = random_vector(rng=rng)
v3 = random_vector(rng=rng)

print(f'v1 shape: {v1.shape}, magnitude: {magnitude(v1):.6f}')
print(f'v2 shape: {v2.shape}, magnitude: {magnitude(v2):.6f}')

# Cosine similarity between random vectors
sim_12 = cosine_similarity(v1, v2)
sim_13 = cosine_similarity(v1, v3)
sim_11 = cosine_similarity(v1, v1)

print(f'\nCosine Similarities:')
print(f'  v1·v1 = {sim_11:.6f} (self, should be 1.0)')
print(f'  v1·v2 = {sim_12:.6f} (random pair)')
print(f'  v1·v3 = {sim_13:.6f} (random pair)')

# Euclidean distance
dist_12 = euclidean_distance(v1, v2)
print(f'\nEuclidean distance v1↔v2: {dist_12:.6f}')

In [ ]:
# Vector arithmetic
v_sum = add(v1, v2)
v_diff = subtract(v1, v2)
v_scaled = scale(v1, 2.5)
v_mid = lerp(v1, v2, 0.5)
v_center = centroid([v1, v2, v3])

print('Vector Arithmetic:')
print(f'  |v1 + v2|  = {magnitude(v_sum):.4f}')
print(f'  |v1 - v2|  = {magnitude(v_diff):.4f}')
print(f'  |2.5 * v1| = {magnitude(v_scaled):.4f}')
print(f'  |lerp(0.5)| = {magnitude(v_mid):.4f}')
print(f'  |centroid|   = {magnitude(v_center):.4f}')

# The centroid should be close to all three vectors
print(f'\nCentroid similarity to each vector:')
for i, v in enumerate([v1, v2, v3], 1):
    print(f'  centroid·v{i} = {cosine_similarity(v_center, v):.4f}')

---
## 2. Similarity Distribution in 384D Space

In high-dimensional space, random unit vectors tend to be near-orthogonal.

In [ ]:
# Generate many random pairs and measure similarity
n_pairs = 5000
similarities = []
for _ in range(n_pairs):
    a = random_vector(rng=rng)
    b = random_vector(rng=rng)
    similarities.append(cosine_similarity(a, b))

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=similarities, nbinsx=80,
    marker=dict(color='#6366f1', line=dict(width=0.5, color='white')),
    name='Cosine Similarity'
))
fig.add_vline(x=0, line_dash='dash', line_color='gray')
fig.add_vline(x=DRIFT_THRESHOLD, line_dash='dash', line_color='red',
              annotation_text=f'Drift Threshold ({DRIFT_THRESHOLD})')
fig.update_layout(
    title=f'Cosine Similarity Distribution in {EMBEDDING_DIM}D Space ({n_pairs:,} pairs)',
    xaxis_title='Cosine Similarity', yaxis_title='Count',
    height=400,
)
fig.show()

print(f'Mean similarity:   {np.mean(similarities):.4f}')
print(f'Std deviation:     {np.std(similarities):.4f}')
print(f'Min:               {np.min(similarities):.4f}')
print(f'Max:               {np.max(similarities):.4f}')

---
## 3. Vector Memory — Store, Search, Drift

The RAM-first vector memory with namespace isolation.

In [ ]:
# Initialize memory and store semantic concepts
memory = VectorMemory()

# Simulate concept embeddings (in production, these come from a model)
concepts = {
    'deployment': random_vector(rng=np.random.default_rng(100)),
    'monitoring': random_vector(rng=np.random.default_rng(101)),
    'security':   random_vector(rng=np.random.default_rng(102)),
    'scaling':    random_vector(rng=np.random.default_rng(103)),
    'debugging':  random_vector(rng=np.random.default_rng(104)),
}

# Make 'scaling' similar to 'deployment' for testing
concepts['scaling'] = normalize(concepts['deployment'] + random_vector(rng=rng) * 0.3)

for name, vec in concepts.items():
    memory.store(name, vec.tolist(), {'category': 'ops', 'importance': np.random.randint(1, 10)})

stats = memory.stats()
print(f'Stored {stats.total_vectors} vectors in {stats.namespaces}')
print(f'Memory estimate: {stats.memory_estimate_bytes:,} bytes')

In [ ]:
# Semantic search
query = concepts['deployment']
results = memory.search(query.tolist(), limit=5, min_score=0.0)

print('Search results for "deployment" query:')
print(f'{"Key":15s} {"Score":>8s} {"Metadata"}')
print('-' * 50)
for r in results:
    print(f'{r.key:15s} {r.score:8.4f} {r.metadata}')

In [ ]:
# Drift detection
v_original = concepts['deployment']
v_drifted = normalize(v_original + random_vector(rng=rng) * 2.0)
v_stable = normalize(v_original + random_vector(rng=rng) * 0.1)

drift_drifted = memory.detect_drift(v_original, v_drifted)
drift_stable = memory.detect_drift(v_original, v_stable)

print(f'Drift detection (threshold={DRIFT_THRESHOLD}):')
print(f'  Heavily perturbed: sim={drift_drifted.similarity:.4f}, drifting={drift_drifted.is_drifting}')
print(f'  Slightly perturbed: sim={drift_stable.similarity:.4f}, drifting={drift_stable.is_drifting}')

---
## 4. PCA — Dimensionality Reduction

Reduce 384D embeddings to 2D/3D for visualization using power iteration PCA.

In [ ]:
# Generate clustered vectors for PCA visualization
n_per_cluster = 30
clusters = {
    'Infrastructure': random_vector(rng=np.random.default_rng(10)),
    'AI Models': random_vector(rng=np.random.default_rng(20)),
    'Security': random_vector(rng=np.random.default_rng(30)),
}

all_vectors = []
all_labels = []
for label, center in clusters.items():
    for _ in range(n_per_cluster):
        noisy = normalize(center + rng.standard_normal(EMBEDDING_DIM) * 0.4)
        all_vectors.append(noisy)
        all_labels.append(label)

# PCA to 3D
projected_3d = pca(all_vectors, 3)

# Also to 2D for a flat plot
projected_2d = pca(all_vectors, 2)

print(f'Projected {len(all_vectors)} vectors from {EMBEDDING_DIM}D → 3D and 2D')

In [ ]:
# 3D scatter plot
colors = {'Infrastructure': '#6366f1', 'AI Models': '#22c55e', 'Security': '#ef4444'}

fig = go.Figure()
for label in clusters:
    indices = [i for i, l in enumerate(all_labels) if l == label]
    xs = [float(projected_3d[i][0]) for i in indices]
    ys = [float(projected_3d[i][1]) for i in indices]
    zs = [float(projected_3d[i][2]) for i in indices]
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode='markers',
        name=label, marker=dict(size=5, color=colors[label], opacity=0.8)
    ))

fig.update_layout(
    title=f'Heady™ {EMBEDDING_DIM}D → 3D PCA Projection',
    scene=dict(xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3'),
    height=600,
)
fig.show()

In [ ]:
# 2D scatter plot
fig, ax = plt.subplots(figsize=(10, 8))
for label in clusters:
    indices = [i for i, l in enumerate(all_labels) if l == label]
    xs = [float(projected_2d[i][0]) for i in indices]
    ys = [float(projected_2d[i][1]) for i in indices]
    ax.scatter(xs, ys, label=label, alpha=0.7, s=50, color=colors[label])

ax.set_title(f'Heady™ {EMBEDDING_DIM}D → 2D PCA Projection', fontsize=14)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Persistence — JSON-Lines Storage

In [ ]:
import tempfile, os

# Persist to disk
tmpfile = os.path.join(tempfile.gettempdir(), 'heady_vectors.jsonl')
memory.persist(tmpfile)
file_size = os.path.getsize(tmpfile)
print(f'Persisted to {tmpfile} ({file_size:,} bytes)')

# Load into fresh memory
memory2 = VectorMemory()
count = memory2.load(tmpfile)
print(f'Loaded {count} vectors into fresh memory')

# Verify integrity
for key in concepts:
    orig = memory.get(key)
    loaded = memory2.get(key)
    sim = cosine_similarity(orig.vector, loaded.vector)
    print(f'  {key:15s} → similarity after persist/load: {sim:.6f}')

os.unlink(tmpfile)

---
## 6. Real Embeddings with sentence-transformers

Generate actual 384D embeddings from text using `all-MiniLM-L6-v2`.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dimensional output

    sentences = [
        'Deploy the application to Cloud Run',
        'Launch the service on Google Cloud',
        'Monitor system health and uptime',
        'Check CPU and memory utilization',
        'Encrypt data with post-quantum algorithms',
        'Secure the API with OAuth 2.0 tokens',
        'Train a neural network on the dataset',
        'Fine-tune the language model',
    ]

    embeddings = model.encode(sentences)
    print(f'Generated {len(embeddings)} embeddings of dim {embeddings[0].shape[0]}')

    # Store in Heady memory
    real_memory = VectorMemory(default_namespace='sentences')
    for i, (sent, emb) in enumerate(zip(sentences, embeddings)):
        real_memory.store(f'sent_{i}', emb.tolist(), {'text': sent}, namespace='sentences')

    # Search for semantically similar sentences
    query_embedding = model.encode(['Ship code to production'])[0]
    results = real_memory.search(query_embedding.tolist(), limit=3, min_score=0.3, namespace='sentences')

    print(f'\nQuery: "Ship code to production"')
    print(f'{"Score":>8s}  Text')
    print('-' * 60)
    for r in results:
        print(f'{r.score:8.4f}  {r.metadata["text"]}')

except ImportError:
    print('sentence-transformers not installed — skipping real embedding demo')
    print('Run: pip install sentence-transformers')